# Практична робота №15
## Критерії узгодженості й таблиці спряженості: критерій хі-квадрат
**Виконав:** Хрустовський Павло  
**Група:** ІТ-42  
**Варіант:** 6 (Чернігів)  
**№ у журналі:** 26


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

np.random.seed(42)

base_temp = 8.0
amplitude = 13
city = "Чернігів"

def get_season(m):
    if m in (12, 1, 2):
        return "зима"
    elif m in (3, 4, 5):
        return "весна"
    elif m in (6, 7, 8):
        return "літо"
    return "осінь"

data = []
for yr in [2021, 2022, 2023, 2024]:
    for m in range(1, 13):
        t_cycle = amplitude * np.cos((m - 7) / 12 * 2 * np.pi)
        t_noise = np.random.normal(0, 1.0)
        t_actual = round(base_temp + t_cycle + t_noise, 1)
        diff = t_actual - base_temp
        if diff < -3:
            state = "холодніше"
        elif diff > 3:
            state = "тепліше"
        else:
            state = "звичайно"
        data.append({
            "місто": city, "рік": yr, "місяць": m,
            "температура": t_actual, "сезон": get_season(m),
            "відхилення_від_норми": state
        })

climate = pd.DataFrame(data)
print(climate.head())


      місто   рік  місяць  температура  сезон відхилення_від_норми
0  Чернігів  2021       1         -4.5   зима            холодніше
1  Чернігів  2021       2         -3.4   зима            холодніше
2  Чернігів  2021       3          2.1  весна            холодніше
3  Чернігів  2021       4          9.5  весна             звичайно
4  Чернігів  2021       5         14.3  весна              тепліше


In [2]:
cross_table = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
print("Таблиця спряженості (абсолютні частоти):\n", cross_table)

cross_table_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize='index')
print("\nНормована таблиця (відносні частоти за рядками):\n", cross_table_norm.round(3))


Таблиця спряженості (абсолютні частоти):
 відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                        4        4          4
зима                         0        0         12
літо                         0       12          0
осінь                        4        4          4

Нормована таблиця (відносні частоти за рядками):
 відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                    0.333    0.333      0.333
зима                     0.000    0.000      1.000
літо                     0.000    1.000      0.000
осінь                    0.333    0.333      0.333


In [3]:
chi2_stat, p_val, dof, expected = chi2_contingency(cross_table)
print(f"Критерій хі-квадрат: chi2 = {chi2_stat:.2f}, p-value = {p_val:.4f}, df = {dof}")
print("\nОчікувані теоретичні частоти:")
print(pd.DataFrame(expected, index=cross_table.index, columns=cross_table.columns).round(2))


Критерій хі-квадрат: chi2 = 38.40, p-value = 0.0000, df = 6

Очікувані теоретичні частоти:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0


In [4]:
if (expected < 5).any():
    print("Виявлено клітинки з очікуваною частотою менше 5. Здійснюємо укрупнення категорій.")

climate["відхилення_укрупнене"] = climate["відхилення_від_норми"].replace({"холодніше": "не тепліше", "звичайно": "не тепліше"})

table_grouped = pd.crosstab(climate["сезон"], climate["відхилення_укрупнене"])
chi2_g, p_g, dof_g, expected_g = chi2_contingency(table_grouped)

print("\nУкрупнена таблиця спряженості:\n", table_grouped)
print(f"\nСтатистика після об'єднання: chi2 = {chi2_g:.2f}, p-value = {p_g:.4f}, df = {dof_g}")
print("Очікувані частоти:\n", pd.DataFrame(expected_g, index=table_grouped.index, columns=table_grouped.columns).round(2))


Виявлено клітинки з очікуваною частотою менше 5. Здійснюємо укрупнення категорій.

Укрупнена таблиця спряженості:
 відхилення_укрупнене  не тепліше  тепліше
сезон                                    
весна                          8        4
зима                          12        0
літо                           0       12
осінь                          8        4

Статистика після об'єднання: chi2 = 26.06, p-value = 0.0000, df = 3
Очікувані частоти:
 відхилення_укрупнене  не тепліше  тепліше
сезон                                    
весна                        7.0      5.0
зима                         7.0      5.0
літо                         7.0      5.0
осінь                        7.0      5.0


In [5]:
season_counts = climate["сезон"].value_counts().sort_index()
print("Фактичний розподіл за сезонами:\n", season_counts)

total_count = season_counts.sum()
expected_season = [total_count / 4] * 4

chi_unif, p_unif = chisquare(f_obs=season_counts, f_exp=expected_season)
print(f"\nКритерій узгодженості (рівномірні частки): chi2 = {chi_unif:.2f}, p-value = {p_unif:.4f}")


Фактичний розподіл за сезонами:
 сезон
весна    12
зима     12
літо     12
осінь    12
Name: count, dtype: int64

Критерій узгодженості (рівномірні частки): chi2 = 0.00, p-value = 1.0000


In [6]:
obs_values = season_counts.values
hypothetical_props = np.array([0.25, 0.25, 0.25, 0.25])
exp_values = hypothetical_props * total_count

chi_prop, p_prop = chisquare(f_obs=obs_values, f_exp=exp_values)
print(f"Тест узгодженості для пропорційного розподілу: chi2 = {chi_prop:.2f}, p-value = {p_prop:.4f}")


Тест узгодженості для пропорційного розподілу: chi2 = 0.00, p-value = 1.0000


### Відповіді на контрольні запитання

1. **Чому у формулі статистики $\chi^2$ береться квадрат відхилення та нормується на очікувану частоту?**  
   Квадрат різниці $(O_i - E_i)^2$ гарантує додатність відхилень незалежно від їх знаку (щоб додатні та від'ємні розходження не взаємознищувалися). Ділення на очікувану частоту $E_i$ нормує абсолютну похибку відносно масштабу самої клітинки, надаючи кожному доданку однаковий статистичний внесок за припущенням пуассонівського розподілу частот.

2. **За яким принципом обчислюється очікувана частота клітинки таблиці спряженості?**  
   За формулою $E_{ij} = \frac{R_i \cdot C_j}{N}$, де $R_i$ — сума спостережень у рядку, $C_j$ — сума в стовпці, а $N$ — загальний обсяг вибірки. Ця формула математично виражає спільну ймовірність настання двох незалежних подій: $P(A \cap B) = P(A) \cdot P(B)$.

3. **У чому полягає різниця між відсутністю підстав відхилити $H_0$ ($p > 0.05$) та "доведенням незалежності"?**  
   Високе значення $p$-value лише свідчить, що емпіричні дані не суперечать гіпотезі про незалежність у межах поточної потужності вибірки. Це не є математичним доказом абсолютної відсутності зв'язку, оскільки слабкий зв'язок міг просто залишитися непоміченим через малий обсяг спостережень.

4. **Що необхідно робити, коли очікувані частоти менші за 5?**  
   Для забезпечення коректності апроксимації розподілом $\chi^2$ (правило Кокрана) застосовують агрегацію (укрупнення) суміжних або змістовно близьких категорій рядків або стовпців, або використовують точний тест Фішера.
